# Months: teach the trained voice month names, without retraining on the full set

Runs on a free Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU).

Continues from the voice already trained by `../train_piper_colab.ipynb` (newest checkpoint in
`My Drive/voice_training/<SPEAKER>_run/`, **read only**). It trains on the month clips (`<SPEAKER>_months.zip`,
~2.4 min) plus a small random handful of the main clips (`REPLAY`, taken from the `<SPEAKER>.zip` already on
Drive). That handful keeps the voice from drifting on everything else; it is not a retrain on the 30 min set.

**Nothing old is changed:** `<SPEAKER>.zip` and `<SPEAKER>_run/` are only read. This run writes to
`My Drive/voice_training/<SPEAKER>_months_run/` and exports `hi_IN-<SPEAKER>_months-medium`.

**Before you start:** upload `months/data/<SPEAKER>_months.zip` to `My Drive/voice_training/`.
After a Colab disconnect, re-run from the top; training resumes from the newest checkpoint of this run.

In [ ]:
# ---- settings -------------------------------------------------------------
SPEAKER = "krupal"         # main voice already trained: My Drive/voice_training/<SPEAKER>_run/ (read only)
NAME = f"{SPEAKER}_months" # month clips <NAME>.zip; this run's folder <NAME>_run; voice hi_IN-<NAME>-medium
REPLAY = 30               # random main-set clips mixed in (~4 min) so nothing else gets worse; 0 = months only
EXTRA_EPOCHS = 300        # on top of the main run's epochs; ~6 min of audio, so ~1 min per 10 epochs on a T4
BATCH_SIZE = 16           # T4 has 16 GB; lower to 8 if you get "CUDA out of memory"
SAVE_EVERY = 25           # epochs between checkpoints; nothing can be exported before the first one
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE = '/content/drive/MyDrive/voice_training'
RUN = f'{DRIVE}/{NAME}_run'
os.makedirs(RUN, exist_ok=True)
assert os.path.exists(f'{DRIVE}/{NAME}.zip'), f'upload {NAME}.zip (months/data/) to My Drive/voice_training/ first'
assert REPLAY == 0 or os.path.exists(f'{DRIVE}/{SPEAKER}.zip'), f'{SPEAKER}.zip (main set) not on Drive; set REPLAY = 0'

## 1. Install Piper training (about 3-5 min)

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
!pip install -q scikit-build cython
!test -d /content/piper1-gpl || git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!python3 setup.py build_ext --inplace
!pip install -e .[train]
!./build_monotonic_align.sh
!python3 -c "from piper import espeakbridge; print('espeakbridge successfully imported!')"

## 2. Dataset and the starting checkpoint

Month clips + `REPLAY` random main clips (renamed `r…` so ids can't clash), unpacked locally in `/content/data`;
the zips on Drive are not modified. The start is the newest checkpoint of the main `<SPEAKER>_run`.

In [ ]:
import random, zipfile, shutil
shutil.rmtree('/content/data', ignore_errors=True); os.makedirs('/content/data')
!unzip -q -o {DRIVE}/{NAME}.zip -d /content/data
meta = open('/content/data/metadata.csv').read().strip().splitlines()
print(len(meta), 'month clips')
if REPLAY:
    with zipfile.ZipFile(f'{DRIVE}/{SPEAKER}.zip') as z:  # read only
        main = z.read('metadata.csv').decode().strip().splitlines()
        for line in random.Random(0).sample(main, min(REPLAY, len(main))):
            wav, text = line.split('|', 1)
            open(f'/content/data/wavs/r{wav}', 'wb').write(z.read(f'wavs/{wav}'))
            meta.append(f'r{wav}|{text}')
    print(REPLAY, 'main clips mixed in')
open('/content/data/metadata.csv', 'w').write('\n'.join(meta) + '\n')
!wc -l /content/data/metadata.csv && ls /content/data/wavs | wc -l

def newest(run):
    c = sorted(glob.glob(f'{run}/lightning_logs/version_*/checkpoints/*.ckpt'), key=os.path.getmtime)
    return c[-1] if c else None

BASE = newest(f'{DRIVE}/{SPEAKER}_run')
assert BASE, f'no checkpoint in My Drive/voice_training/{SPEAKER}_run/: train the main voice first'

import torch
BASE_EPOCHS = torch.load(BASE, map_location='cpu', weights_only=False)['epoch']  # epochs already in it
print('base', BASE, 'at epoch', BASE_EPOCHS)

## 3. Train

`--ckpt_path` resumes from a checkpoint (its epoch counter included), so `max_epochs` = base epochs + `EXTRA_EPOCHS`.
If an earlier run of this cell saved checkpoints in `<NAME>_run`, it resumes from the newest of those instead.

In [ ]:
START = newest(RUN) or BASE
print('starting from', START)

!PYTHONPATH=/content/piper1-gpl/src python3 -m piper.train fit \
  --data.voice_name "{NAME}" \
  --data.csv_path /content/data/metadata.csv \
  --data.audio_dir /content/data/wavs/ \
  --model.sample_rate 22050 \
  --data.espeak_voice "hi" \
  --data.cache_dir /content/cache/ \
  --data.config_path {RUN}/config.json \
  --data.batch_size {BATCH_SIZE} \
  --trainer.max_epochs {BASE_EPOCHS + EXTRA_EPOCHS} \
  --trainer.check_val_every_n_epoch {SAVE_EVERY} \
  --trainer.default_root_dir {RUN} \
  --trainer.callbacks+=lightning.pytorch.callbacks.ModelCheckpoint \
  --trainer.callbacks.every_n_epochs={SAVE_EVERY} \
  --trainer.callbacks.save_last=true \
  --ckpt_path "{START}"

## 4. Export the newest checkpoint and listen to the months

In [ ]:
import wave, sys
sys.path.append('/content/piper1-gpl/src')
from IPython.display import Audio, display
ckpt = newest(RUN)
assert ckpt, f'no checkpoint in {RUN}/lightning_logs/version_*/checkpoints/: step 3 stopped before epoch {SAVE_EVERY}; train longer'
print('exporting', ckpt)
VOICE = f'hi_IN-{NAME}-medium'
!python3 -m piper.train.export_onnx --checkpoint "{ckpt}" --output-file /content/{VOICE}.onnx
assert os.path.getsize(f"/content/{VOICE}.onnx") > 10_000_000, "export failed, see the error above"
!cp {RUN}/config.json /content/{VOICE}.onnx.json

from piper import PiperVoice
voice = PiperVoice.load(f'/content/{VOICE}.onnx', config_path=f'/content/{VOICE}.onnx.json')
TESTS = [  # months in sentences the bot says, plus a normal line to check nothing else got worse
    "जनवरी, फ़रवरी, मार्च, अप्रैल, मई, जून।",
    "जुलाई, अगस्त, सितंबर, अक्टूबर, नवंबर, दिसंबर।",
    "शनिवार, 28 नवंबर को हमारे पास स्लॉट है, क्या आप उस दिन अवेलेबल हैं?",
    "आपकी विज़िट बुधवार, 2 दिसंबर के लिए कन्फ़र्म हो गई है।",
    "ठीक है, हम 14 फ़रवरी तक आपकी सफ़ाई रोक कर रखेंगे।",
    "मंगलवार, 12 अक्टूबर या गुरुवार, 14 अक्टूबर, कौन सा दिन ठीक रहेगा?",
    "क्या आपके पास ए-टाइप लैडर है, जिससे हीरो छत तक पहुँच सकें?",
]
for i, t in enumerate(TESTS):
    with wave.open(f'/content/test_{i}.wav', 'wb') as w:
        voice.synthesize_wav(t, w)
    print(t); display(Audio(f'/content/test_{i}.wav'))

## 5. Save the voice to Drive

In [ ]:
!cp /content/{VOICE}.onnx /content/{VOICE}.onnx.json {DRIVE}/
print(f'Download {VOICE}.onnx and {VOICE}.onnx.json from My Drive/voice_training/')